# 🐼 Panda AI — One-Click Deploy
Deploy the full Panda AI gateway (API + Dashboard) on Colab.
**Runtime → Run all (Ctrl+F9)**

## 1️⃣ Install

In [ ]:
# Node 20 (binary — Colab apt gives Node 12 which is too old)
!curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1
!pip install -q -r requirements.txt --root-user-action=ignore 2>&1 | grep -v WARNING | tail -2
!patchright install chromium 2>&1 | tail -1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf && mv /tmp/cf /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, sys
v = subprocess.check_output(['node', '--version']).decode().strip()
assert v.startswith('v20'), f'Need Node 20, got {v}'
print(f'✅ Node {v} | Python {sys.version.split()[0]}')

## 2️⃣ Clone

In [ ]:
import os
os.chdir('/')
!rm -rf /content/Panda-Ai
!git clone -q https://github.com/ferelking242/Panda-Ai.git /content/Panda-Ai
os.chdir('/content/Panda-Ai')
# Remove bun.lock at root — it confuses Next.js turbopack workspace detection
!rm -f bun.lock bun.lockb
with open('.env', 'w') as f: f.write('PROVIDER=chatgpt\nHEADLESS=true\nAPI_HOST=0.0.0.0\nAPI_PORT=8000\nPOOL_SIZE=1\nLOG_LEVEL=INFO\n')
print('✅ Repo ready (bun.lock removed)')

## 3️⃣ Build Dashboard

In [ ]:
os.chdir('/content/Panda-Ai/dashboard')
!npm install --no-audit --no-fund 2>&1 | tail -3
!npm run build 2>&1 | tail -8
# Verify build output exists
import os
assert os.path.exists('.next/BUILD_ID'), 'Build failed — no .next/BUILD_ID found'
os.chdir('/content/Panda-Ai')
print('✅ Dashboard built successfully')

## 4️⃣ Start

In [ ]:
import subprocess, time, os, sys

!fuser -k 8000/tcp 2>/dev/null || true
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(2)

env_base = {**os.environ, 'PYTHONUNBUFFERED': '1'}

api_log = open('/tmp/api.log', 'w')
api_proc = subprocess.Popen(
    [sys.executable, '-m', 'src.api.server'],
    cwd='/content/Panda-Ai', env=env_base,
    stdout=api_log, stderr=subprocess.STDOUT
)
print(f'🚀 API starting...')

dash_log = open('/tmp/dash.log', 'w')
dash_env = {**env_base, 'PORT': '5000', 'API_ORIGIN': 'http://127.0.0.1:8000', 'NODE_ENV': 'production'}
dash_proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='/content/Panda-Ai/dashboard', env=dash_env,
    stdout=dash_log, stderr=subprocess.STDOUT
)
print(f'📊 Dashboard starting...')

import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3)
        print(f'✅ API healthy')
        break
    except: pass

for i in range(15):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://127.0.0.1:5000', timeout=3)
        print(f'✅ Dashboard healthy')
        break
    except: pass
else:
    print('⚠️ Dashboard not responding — debug below:')
    !tail -5 /tmp/dash.log

## 5️⃣ Token + URLs

In [ ]:
import json, urllib.request, re, threading

try:
    req = urllib.request.Request(
        'http://127.0.0.1:8000/api/dashboard/token/generate',
        data=json.dumps({'name': 'colab', 'scope': ['*']}).encode(),
        headers={'Content-Type': 'application/json'}, method='POST'
    )
    api_token = json.loads(urllib.request.urlopen(req).read())['token']
except:
    import secrets; api_token = 'pnd_' + secrets.token_hex(16)

urls = {}
def tunnel(port, name):
    p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://localhost:{port}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m: urls[name] = m.group(0); print(f'  🔗 {name}: {m.group(0)}'); break
threading.Thread(target=tunnel, args=(8000,'API'), daemon=True).start()
threading.Thread(target=tunnel, args=(5000,'Dashboard'), daemon=True).start()
for _ in range(30): time.sleep(1)
    if len(urls) >= 2: break

print(f'\n═══════════════════════════════════════════')
print(f'  🐼 PANDA AI DEPLOYED')
print(f'═══════════════════════════════════════════')
if 'API' in urls: print(f'  🤖 API:      {urls["API"]}/v1')
if 'Dashboard' in urls: print(f'  📊 Dashboard: {urls["Dashboard"]}')
print(f'  🔑 Token: {api_token}')
print(f'═══════════════════════════════════════════')

## 📋 Quick Test

In [ ]:
import urllib.request, json
base = 'http://127.0.0.1:8000'
print(f'✅ Health: {json.loads(urllib.request.urlopen(f"{base}/healthz").read())}')
req = urllib.request.Request(f'{base}/v1/models', headers={'Authorization': f'Bearer {api_token}'})
models = json.loads(urllib.request.urlopen(req).read())
print(f'✅ Models: {[m["id"] for m in models["data"][:5]]}')
print(f'\n✅ base_url="{urls.get("API","?")}/v1" api_key="{api_token}"')

## 🔧 Debug

In [ ]:
print('=== API LOG ==='); !tail -15 /tmp/api.log
print('\n=== DASHBOARD LOG ==='); !tail -15 /tmp/dash.log
print('\n=== NODE ==='); !node --version